# Stress Prediction v8 — Single Feature A/B Test

이번 버전은 기존 최고 구조를 유지하면서 **파생 피처를 한 번에 하나씩만 추가**해 효과를 측정합니다.

진행 순서:

1. 원본 Baseline CV 재현
2. `BMI`, `맥압`, `평균동맥압`, `혈압비`, `결측 플래그`, 범주 조합 2개를 각각 단독 추가
3. Baseline보다 좋아진 피처만 2차 조합
4. 가장 좋은 피처 조합에서 Tree quantile, Pair quantile, Pair weight, rounding, override 탐색
5. 서로 다른 성격의 제출 후보 최대 5개 생성

> 피처를 한꺼번에 많이 추가하지 않으므로 어떤 피처가 실제로 도움 되는지 명확하게 확인할 수 있습니다.


In [1]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_SPLITS = 5

# 1차/2차 A/B 테스트용
SCREEN_N_ESTIMATORS = 700

# 최종 탐색 및 제출용
FINAL_N_ESTIMATORS = 1200

BASE_TREE_QUANTILE = 0.52

TREE_QUANTILES = [0.50, 0.52, 0.54]
PAIR_QUANTILES = [0.46, 0.48, 0.50]
PAIR_WEIGHTS = [0.15, 0.20, 0.25]
OVERRIDE_THRESHOLDS = [0.15, 0.18, 0.20, None]
ROUND_OPTIONS = [2, None]

WINSORIZE_COLUMNS = [
    "mean_working",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
]

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FALLBACK = Path("/mnt/data/train(2).csv")
TEST_FALLBACK = Path("/mnt/data/test(2).csv")

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


DATA_DIR: /Users/bitsaem/Desktop/stress_project/open
OUTPUT_DIR: /Users/bitsaem/Desktop/stress_project/8_6


## 1. 데이터 로드


In [2]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

if not train_path.exists() and TRAIN_FALLBACK.exists():
    train_path = TRAIN_FALLBACK

if not test_path.exists() and TEST_FALLBACK.exists():
    test_path = TEST_FALLBACK

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

y = train_raw[TARGET].reset_index(drop=True)

X_raw = (
    train_raw
    .drop(columns=[TARGET, ID_COLUMN])
    .reset_index(drop=True)
)

X_test_raw = (
    test_raw
    .drop(columns=[ID_COLUMN])
    .reset_index(drop=True)
)

numeric_raw_columns = X_raw.select_dtypes(
    include=["number"]
).columns.tolist()

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)
display(train_raw.head())


Train shape: (3000, 18)
Test shape : (3000, 17)


,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


## 2. Baseline 및 단일 파생 피처 함수


In [3]:
def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series,
) -> pd.Series:
    return numerator / (denominator.abs() + 1e-6)


def add_baseline_features(frame: pd.DataFrame) -> pd.DataFrame:
    # 기존 최고 구조에서 사용하던 기본 파생변수
    result = frame.copy()

    height_m = result["height"] / 100.0

    result["bmi"] = safe_divide(
        result["weight"],
        height_m.pow(2),
    )

    result["cholesterol_glucose_ratio"] = safe_divide(
        result["cholesterol"],
        result["glucose"],
    )

    return result


def add_candidate_feature(
    frame: pd.DataFrame,
    feature_name: str,
) -> pd.DataFrame:
    result = add_baseline_features(frame)

    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    if feature_name == "none":
        return result

    if feature_name == "pulse_pressure":
        result["pulse_pressure"] = sbp - dbp

    elif feature_name == "mean_arterial_pressure":
        result["mean_arterial_pressure"] = (
            sbp + 2.0 * dbp
        ) / 3.0

    elif feature_name == "blood_pressure_ratio":
        result["blood_pressure_ratio"] = safe_divide(
            sbp,
            dbp,
        )

    elif feature_name == "missing_flag":
        result["missing_flag"] = (
            frame.isna().any(axis=1).astype(int)
        )

    elif feature_name == "activity_sleep_cross":
        result["activity_sleep_cross"] = (
            result["activity"]
            .fillna("__MISSING__")
            .astype(str)
            + "__"
            + result["sleep_pattern"]
            .fillna("__MISSING__")
            .astype(str)
        )

    elif feature_name == "medical_family_cross":
        result["medical_family_cross"] = (
            result["medical_history"]
            .fillna("__MISSING__")
            .astype(str)
            + "__"
            + result["family_medical_history"]
            .fillna("__MISSING__")
            .astype(str)
        )

    elif feature_name == "bmi_duplicate_test":
        # BMI는 baseline에 이미 있으므로
        # 단독 후보 목록에서 baseline과 동일함을 확인하기 위한 항목
        pass

    else:
        raise ValueError(
            f"알 수 없는 feature_name: {feature_name}"
        )

    return result


def add_feature_combination(
    frame: pd.DataFrame,
    features: tuple[str, ...],
) -> pd.DataFrame:
    result = add_baseline_features(frame)

    for feature_name in features:
        temp = add_candidate_feature(frame, feature_name)

        new_columns = [
            col for col in temp.columns
            if col not in result.columns
        ]

        for col in new_columns:
            result[col] = temp[col]

    return result


SINGLE_FEATURES = [
    "none",
    "pulse_pressure",
    "mean_arterial_pressure",
    "blood_pressure_ratio",
    "missing_flag",
    "activity_sleep_cross",
    "medical_family_cross",
]

print("단일 실험 후보:", SINGLE_FEATURES)


단일 실험 후보: ['none', 'pulse_pressure', 'mean_arterial_pressure', 'blood_pressure_ratio', 'missing_flag', 'activity_sleep_cross', 'medical_family_cross']


## 3. 기존 ExtraTrees / Pair / Override 함수


In [4]:
def compute_iqr_bounds(
    train_frame: pd.DataFrame,
    columns: list[str],
):
    bounds = {}

    for col in columns:
        q1, q3 = train_frame[col].quantile(
            [0.25, 0.75]
        )
        iqr = q3 - q1

        bounds[col] = (
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr,
        )

    return bounds


def apply_winsorize(
    frame: pd.DataFrame,
    bounds,
):
    result = frame.copy()

    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(
            lower,
            upper,
        )

    return result


def add_feature_copies(
    frame: pd.DataFrame,
):
    result = frame.copy()

    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue

        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = (
                result[feature]
            )

    return result


def make_preprocessor(
    frame: pd.DataFrame,
):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numerical = frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    return ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing",
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
            numerical,
        ),
    ])


def tree_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    n_estimators: int,
):
    # 기존 최고 구조 유지: gender 제거 + feature copy
    train_input = train_features.drop(
        columns=["gender"],
        errors="ignore",
    )
    val_input = val_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    train_input = add_feature_copies(train_input)
    val_input = add_feature_copies(val_input)

    preprocessor = make_preprocessor(train_input)

    train_matrix = preprocessor.fit_transform(
        train_input
    )
    val_matrix = preprocessor.transform(
        val_input
    )

    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    model.fit(
        train_matrix,
        target,
    )

    return np.column_stack([
        tree.predict(val_matrix)
        for tree in model.estimators_
    ])


def empirical_rank_transform(
    train_values,
    test_values,
):
    median = np.nanmedian(train_values)

    train_values = np.nan_to_num(
        train_values,
        nan=median,
    )
    test_values = np.nan_to_num(
        test_values,
        nan=median,
    )

    sorted_train = np.sort(train_values)

    train_rank = (
        np.searchsorted(
            sorted_train,
            train_values,
            side="right",
        )
        / len(train_values)
    )

    test_rank = (
        np.searchsorted(
            sorted_train,
            test_values,
            side="right",
        )
        / len(train_values)
    )

    return (
        train_rank.astype(np.float32),
        test_rank.astype(np.float32),
    )


def pair_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
):
    train_rank = {}
    val_rank = {}

    for col in PAIR_COLUMNS:
        train_rank[col], val_rank[col] = (
            empirical_rank_transform(
                train_features[col].to_numpy(dtype=float),
                val_features[col].to_numpy(dtype=float),
            )
        )

    target_values = target.to_numpy(dtype=float)
    pair_targets = []

    for first, second in combinations(
        PAIR_COLUMNS,
        2,
    ):
        distance = (
            np.abs(
                val_rank[first][:, None]
                - train_rank[first][None, :]
            )
            + np.abs(
                val_rank[second][:, None]
                - train_rank[second][None, :]
            )
        )

        nearest_idx = np.argmin(
            distance,
            axis=1,
        )

        pair_targets.append(
            target_values[nearest_idx]
        )

    return np.column_stack(pair_targets)


def nearest_neighbor_lookup(
    train_raw_frame: pd.DataFrame,
    val_raw_frame: pd.DataFrame,
    target: pd.Series,
):
    train_numeric = train_raw_frame[
        numeric_raw_columns
    ].copy()

    val_numeric = val_raw_frame[
        numeric_raw_columns
    ].copy()

    medians = train_numeric.median()

    train_numeric = train_numeric.fillna(
        medians
    )
    val_numeric = val_numeric.fillna(
        medians
    )

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_numeric
    )
    val_scaled = scaler.transform(
        val_numeric
    )

    tree = cKDTree(train_scaled)

    nearest_dist, nearest_idx = tree.query(
        val_scaled,
        k=1,
    )

    nearest_target = (
        target.to_numpy(dtype=float)[nearest_idx]
    )

    return nearest_dist, nearest_target


def apply_postprocess(
    prediction,
    round_decimals,
    override_threshold,
    nearest_dist,
    nearest_target,
):
    result = np.clip(
        prediction.copy(),
        0.0,
        1.0,
    )

    if round_decimals is not None:
        result = np.round(
            result,
            round_decimals,
        )

    if override_threshold is not None:
        mask = nearest_dist < override_threshold
        replacement = nearest_target[mask].copy()

        if round_decimals is not None:
            replacement = np.round(
                replacement,
                round_decimals,
            )

        result[mask] = np.clip(
            replacement,
            0.0,
            1.0,
        )

    return result


## 4. 1차 단일 피처 A/B 테스트

모든 실험은 동일한 CV split, 동일한 ExtraTrees 설정으로 비교됩니다.


In [5]:
cv_splits = list(
    KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    ).split(X_raw)
)

single_rows = []
single_oof = {}

for feature_name in SINGLE_FEATURES:
    oof_pred = np.zeros(
        len(y),
        dtype=float,
    )

    for fold, (train_idx, val_idx) in enumerate(
        cv_splits,
        start=1,
    ):
        train_fold_raw = (
            X_raw.iloc[train_idx]
            .reset_index(drop=True)
        )

        val_fold_raw = (
            X_raw.iloc[val_idx]
            .reset_index(drop=True)
        )

        y_train = (
            y.iloc[train_idx]
            .reset_index(drop=True)
        )

        bounds = compute_iqr_bounds(
            train_fold_raw,
            WINSORIZE_COLUMNS,
        )

        train_w = apply_winsorize(
            train_fold_raw,
            bounds,
        )

        val_w = apply_winsorize(
            val_fold_raw,
            bounds,
        )

        train_features = add_candidate_feature(
            train_w,
            feature_name,
        )

        val_features = add_candidate_feature(
            val_w,
            feature_name,
        )

        tree_matrix = tree_prediction_matrix(
            train_features,
            val_features,
            y_train,
            n_estimators=SCREEN_N_ESTIMATORS,
        )

        pred = np.quantile(
            tree_matrix,
            BASE_TREE_QUANTILE,
            axis=1,
        )

        pred = np.round(
            np.clip(pred, 0.0, 1.0),
            2,
        )

        oof_pred[val_idx] = pred

        print(
            f"[{feature_name}] Fold "
            f"{fold}/{N_SPLITS} 완료"
        )

    mae = mean_absolute_error(
        y,
        oof_pred,
    )

    single_rows.append({
        "feature": feature_name,
        "cv_mae": mae,
    })

    single_oof[feature_name] = oof_pred

    print(
        f"{feature_name}: CV MAE={mae:.8f}\n"
    )

single_df = pd.DataFrame(
    single_rows
)

baseline_mae = float(
    single_df.loc[
        single_df["feature"] == "none",
        "cv_mae",
    ].iloc[0]
)

single_df["improvement_vs_baseline"] = (
    baseline_mae - single_df["cv_mae"]
)

single_df = (
    single_df
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(single_df)


[none] Fold 1/5 완료
[none] Fold 2/5 완료
[none] Fold 3/5 완료
[none] Fold 4/5 완료
[none] Fold 5/5 완료
none: CV MAE=0.14811333

[pulse_pressure] Fold 1/5 완료
[pulse_pressure] Fold 2/5 완료
[pulse_pressure] Fold 3/5 완료
[pulse_pressure] Fold 4/5 완료
[pulse_pressure] Fold 5/5 완료
pulse_pressure: CV MAE=0.14752667

[mean_arterial_pressure] Fold 1/5 완료
[mean_arterial_pressure] Fold 2/5 완료
[mean_arterial_pressure] Fold 3/5 완료
[mean_arterial_pressure] Fold 4/5 완료
[mean_arterial_pressure] Fold 5/5 완료
mean_arterial_pressure: CV MAE=0.14812333

[blood_pressure_ratio] Fold 1/5 완료
[blood_pressure_ratio] Fold 2/5 완료
[blood_pressure_ratio] Fold 3/5 완료
[blood_pressure_ratio] Fold 4/5 완료
[blood_pressure_ratio] Fold 5/5 완료
blood_pressure_ratio: CV MAE=0.14749667

[missing_flag] Fold 1/5 완료
[missing_flag] Fold 2/5 완료
[missing_flag] Fold 3/5 완료
[missing_flag] Fold 4/5 완료
[missing_flag] Fold 5/5 완료
missing_flag: CV MAE=0.14812333

[activity_sleep_cross] Fold 1/5 완료
[activity_sleep_cross] Fold 2/5 완료
[activity_sleep_cr

,feature,cv_mae,improvement_vs_baseline
0,blood_pressure_ratio,0.147497,0.000617
1,pulse_pressure,0.147527,0.000587
2,none,0.148113,0.000000
3,mean_arterial_pressure,0.148123,-0.000010
4,missing_flag,0.148123,-0.000010
5,medical_family_cross,0.148460,-0.000347
6,activity_sleep_cross,0.149140,-0.001027


## 5. 2차 조합 실험

Baseline보다 좋아진 단일 피처만 조합합니다.

개선 피처가 없으면 가장 성능이 좋았던 상위 3개를 사용해 2개 조합만 비교합니다.


In [6]:
MIN_SINGLE_IMPROVEMENT = 0.00005

improved_features = (
    single_df.loc[
        (
            single_df["feature"] != "none"
        )
        & (
            single_df["improvement_vs_baseline"]
            >= MIN_SINGLE_IMPROVEMENT
        ),
        "feature",
    ]
    .tolist()
)

if not improved_features:
    improved_features = (
        single_df.loc[
            single_df["feature"] != "none",
            "feature",
        ]
        .head(3)
        .tolist()
    )

print(
    "2차 조합 후보 피처:",
    improved_features,
)

combination_candidates = []

for size in [2, 3]:
    if len(improved_features) >= size:
        combination_candidates.extend(
            list(
                combinations(
                    improved_features,
                    size,
                )
            )
        )

# 계산량 방지를 위해 최대 10개 조합
combination_candidates = combination_candidates[:10]

print(
    "2차 조합 수:",
    len(combination_candidates),
)

combination_rows = []
combination_oof = {}

for feature_combo in combination_candidates:
    oof_pred = np.zeros(
        len(y),
        dtype=float,
    )

    for fold, (train_idx, val_idx) in enumerate(
        cv_splits,
        start=1,
    ):
        train_fold_raw = (
            X_raw.iloc[train_idx]
            .reset_index(drop=True)
        )

        val_fold_raw = (
            X_raw.iloc[val_idx]
            .reset_index(drop=True)
        )

        y_train = (
            y.iloc[train_idx]
            .reset_index(drop=True)
        )

        bounds = compute_iqr_bounds(
            train_fold_raw,
            WINSORIZE_COLUMNS,
        )

        train_w = apply_winsorize(
            train_fold_raw,
            bounds,
        )

        val_w = apply_winsorize(
            val_fold_raw,
            bounds,
        )

        train_features = add_feature_combination(
            train_w,
            feature_combo,
        )

        val_features = add_feature_combination(
            val_w,
            feature_combo,
        )

        tree_matrix = tree_prediction_matrix(
            train_features,
            val_features,
            y_train,
            n_estimators=SCREEN_N_ESTIMATORS,
        )

        pred = np.quantile(
            tree_matrix,
            BASE_TREE_QUANTILE,
            axis=1,
        )

        pred = np.round(
            np.clip(pred, 0.0, 1.0),
            2,
        )

        oof_pred[val_idx] = pred

    combo_name = " + ".join(feature_combo)

    mae = mean_absolute_error(
        y,
        oof_pred,
    )

    combination_rows.append({
        "features": combo_name,
        "feature_tuple": feature_combo,
        "cv_mae": mae,
        "improvement_vs_baseline": (
            baseline_mae - mae
        ),
    })

    combination_oof[feature_combo] = oof_pred

    print(
        f"{combo_name}: CV MAE={mae:.8f}"
    )

if combination_rows:
    combination_df = (
        pd.DataFrame(combination_rows)
        .sort_values("cv_mae")
        .reset_index(drop=True)
    )

    display(combination_df)
else:
    combination_df = pd.DataFrame()
    print("2차 조합 후보가 없습니다.")


2차 조합 후보 피처: ['blood_pressure_ratio', 'pulse_pressure']
2차 조합 수: 1
blood_pressure_ratio + pulse_pressure: CV MAE=0.14780000


,features,feature_tuple,cv_mae,improvement_vs_baseline
0,blood_pressure_ratio + pulse_pressure,"(blood_pressure_ratio, pulse_pressure)",0.1478,0.000313


## 6. 최종 피처 조합 선택


In [7]:
best_single_row = single_df.iloc[0]

BEST_FEATURES = tuple()

best_feature_mae = baseline_mae
best_feature_source = "baseline"

if (
    best_single_row["feature"] != "none"
    and best_single_row["cv_mae"] < best_feature_mae
):
    BEST_FEATURES = (
        str(best_single_row["feature"]),
    )

    best_feature_mae = float(
        best_single_row["cv_mae"]
    )

    best_feature_source = "single"

if len(combination_df):
    best_combo_row = combination_df.iloc[0]

    if best_combo_row["cv_mae"] < best_feature_mae:
        BEST_FEATURES = tuple(
            best_combo_row["feature_tuple"]
        )

        best_feature_mae = float(
            best_combo_row["cv_mae"]
        )

        best_feature_source = "combination"

# 개선 폭이 너무 작으면 baseline 유지
MIN_FINAL_IMPROVEMENT = 0.00005

if baseline_mae - best_feature_mae < MIN_FINAL_IMPROVEMENT:
    BEST_FEATURES = tuple()
    best_feature_mae = baseline_mae
    best_feature_source = "baseline_safe"

print("선택된 최종 피처:", BEST_FEATURES)
print("선택 출처:", best_feature_source)
print("Baseline MAE:", baseline_mae)
print("선택 피처 MAE:", best_feature_mae)
print(
    "개선 폭:",
    baseline_mae - best_feature_mae,
)


선택된 최종 피처: ('blood_pressure_ratio',)
선택 출처: single
Baseline MAE: 0.14811333333333335
선택 피처 MAE: 0.14749666666666666
개선 폭: 0.000616666666666682


## 7. 최종 피처로 Pair·Quantile·Override 탐색


In [8]:
fold_cache = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits,
    start=1,
):
    train_fold_raw = (
        X_raw.iloc[train_idx]
        .reset_index(drop=True)
    )

    val_fold_raw = (
        X_raw.iloc[val_idx]
        .reset_index(drop=True)
    )

    y_train = (
        y.iloc[train_idx]
        .reset_index(drop=True)
    )

    bounds = compute_iqr_bounds(
        train_fold_raw,
        WINSORIZE_COLUMNS,
    )

    train_w = apply_winsorize(
        train_fold_raw,
        bounds,
    )

    val_w = apply_winsorize(
        val_fold_raw,
        bounds,
    )

    train_features = add_feature_combination(
        train_w,
        BEST_FEATURES,
    )

    val_features = add_feature_combination(
        val_w,
        BEST_FEATURES,
    )

    tree_matrix = tree_prediction_matrix(
        train_features,
        val_features,
        y_train,
        n_estimators=FINAL_N_ESTIMATORS,
    )

    # Pair에는 기존 baseline feature만 사용
    train_pair = add_baseline_features(
        train_w
    )

    val_pair = add_baseline_features(
        val_w
    )

    pair_matrix = pair_prediction_matrix(
        train_pair,
        val_pair,
        y_train,
    )

    nearest_dist, nearest_target = (
        nearest_neighbor_lookup(
            train_fold_raw,
            val_fold_raw,
            y_train,
        )
    )

    fold_cache.append({
        "val_idx": val_idx,
        "tree_matrix": tree_matrix,
        "pair_matrix": pair_matrix,
        "nearest_dist": nearest_dist,
        "nearest_target": nearest_target,
    })

    print(
        f"Fold {fold}/{N_SPLITS} 캐시 완료"
    )


Fold 1/5 캐시 완료
Fold 2/5 캐시 완료
Fold 3/5 캐시 완료
Fold 4/5 캐시 완료
Fold 5/5 캐시 완료


In [9]:
search_rows = []

for tree_q in TREE_QUANTILES:
    for pair_q in PAIR_QUANTILES:
        for pair_weight in PAIR_WEIGHTS:
            for decimals in ROUND_OPTIONS:
                for threshold in OVERRIDE_THRESHOLDS:
                    oof_pred = np.zeros(
                        len(y),
                        dtype=float,
                    )

                    for cache in fold_cache:
                        tree_pred = np.quantile(
                            cache["tree_matrix"],
                            tree_q,
                            axis=1,
                        )

                        pair_pred = np.quantile(
                            cache["pair_matrix"],
                            pair_q,
                            axis=1,
                        )

                        blended = (
                            (1.0 - pair_weight)
                            * tree_pred
                            + pair_weight
                            * pair_pred
                        )

                        final_pred = apply_postprocess(
                            blended,
                            decimals,
                            threshold,
                            cache["nearest_dist"],
                            cache["nearest_target"],
                        )

                        oof_pred[
                            cache["val_idx"]
                        ] = final_pred

                    mae = mean_absolute_error(
                        y,
                        oof_pred,
                    )

                    search_rows.append({
                        "cv_mae": mae,
                        "tree_q": tree_q,
                        "pair_q": pair_q,
                        "pair_weight": pair_weight,
                        "round_decimals": decimals,
                        "override_threshold": threshold,
                        "selected_features": (
                            "baseline"
                            if not BEST_FEATURES
                            else " + ".join(BEST_FEATURES)
                        ),
                    })

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(search_df.head(30))
print(
    "최적 최종 OOF MAE:",
    float(search_df.iloc[0]["cv_mae"]),
)


,cv_mae,tree_q,pair_q,pair_weight,round_decimals,override_threshold,selected_features
0,0.147483,0.54,0.48,0.25,2.0,0.20,blood_pressure_ratio
1,0.147483,0.54,0.48,0.25,2.0,0.18,blood_pressure_ratio
2,0.147483,0.54,0.48,0.25,2.0,0.15,blood_pressure_ratio
3,0.147506,0.54,0.46,0.25,NaN,0.15,blood_pressure_ratio
4,0.147506,0.54,0.46,0.25,NaN,0.18,blood_pressure_ratio
5,0.147506,0.54,0.46,0.25,NaN,0.20,blood_pressure_ratio
6,0.147512,0.54,0.48,0.25,NaN,0.20,blood_pressure_ratio
7,0.147512,0.54,0.48,0.25,NaN,0.18,blood_pressure_ratio
8,0.147512,0.54,0.48,0.25,NaN,0.15,blood_pressure_ratio
9,0.147517,0.52,0.48,0.15,2.0,0.15,blood_pressure_ratio


최적 최종 OOF MAE: 0.14748333333333333


## 8. 제출 후보 생성


In [10]:
def row_to_config(row):
    return {
        "tree_q": float(row["tree_q"]),
        "pair_q": float(row["pair_q"]),
        "pair_weight": float(row["pair_weight"]),
        "round_decimals": (
            None
            if pd.isna(row["round_decimals"])
            else int(row["round_decimals"])
        ),
        "override_threshold": (
            None
            if pd.isna(row["override_threshold"])
            else float(row["override_threshold"])
        ),
        "cv_mae": float(row["cv_mae"]),
    }


candidate_rows = [
    ("cv_best", search_df.iloc[0]),
]

safe_rows = search_df[
    (search_df["tree_q"] == 0.52)
    & (search_df["pair_q"] == 0.48)
    & (search_df["pair_weight"] == 0.20)
    & (search_df["round_decimals"] == 2)
]
if len(safe_rows):
    candidate_rows.append(
        ("safe", safe_rows.iloc[0])
    )

no_override_rows = search_df[
    search_df["override_threshold"].isna()
]
if len(no_override_rows):
    candidate_rows.append(
        (
            "no_override",
            no_override_rows.iloc[0],
        )
    )

tree_heavy_rows = search_df[
    search_df["pair_weight"]
    == min(PAIR_WEIGHTS)
]
if len(tree_heavy_rows):
    candidate_rows.append(
        (
            "tree_heavy",
            tree_heavy_rows.iloc[0],
        )
    )

pair_heavy_rows = search_df[
    search_df["pair_weight"]
    == max(PAIR_WEIGHTS)
]
if len(pair_heavy_rows):
    candidate_rows.append(
        (
            "pair_heavy",
            pair_heavy_rows.iloc[0],
        )
    )

FINAL_CONFIGS = []
seen = set()

for name, row in candidate_rows:
    config = row_to_config(row)

    key = (
        config["tree_q"],
        config["pair_q"],
        config["pair_weight"],
        config["round_decimals"],
        config["override_threshold"],
    )

    if key not in seen:
        seen.add(key)
        FINAL_CONFIGS.append(
            (name, config)
        )

FINAL_CONFIGS = FINAL_CONFIGS[:5]

print("최종 제출 후보")
for name, config in FINAL_CONFIGS:
    print(name, config)


최종 제출 후보
cv_best {'tree_q': 0.54, 'pair_q': 0.48, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14748333333333333}
safe {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.2, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14752}
no_override {'tree_q': 0.5, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14756000000000002}
tree_heavy {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14751666666666666}


In [11]:
if not sample_path.exists():
    submission_template = pd.DataFrame({
        ID_COLUMN: test_raw[ID_COLUMN],
        TARGET: 0.0,
    })
else:
    submission_template = pd.read_csv(
        sample_path
    )

bounds = compute_iqr_bounds(
    X_raw,
    WINSORIZE_COLUMNS,
)

train_w = apply_winsorize(
    X_raw,
    bounds,
)

test_w = apply_winsorize(
    X_test_raw,
    bounds,
)

train_features = add_feature_combination(
    train_w,
    BEST_FEATURES,
)

test_features = add_feature_combination(
    test_w,
    BEST_FEATURES,
)

tree_matrix_test = tree_prediction_matrix(
    train_features,
    test_features,
    y,
    n_estimators=FINAL_N_ESTIMATORS,
)

train_pair = add_baseline_features(
    train_w
)

test_pair = add_baseline_features(
    test_w
)

pair_matrix_test = pair_prediction_matrix(
    train_pair,
    test_pair,
    y,
)

nearest_dist_test, nearest_target_test = (
    nearest_neighbor_lookup(
        X_raw,
        X_test_raw,
        y,
    )
)

feature_label = (
    "baseline"
    if not BEST_FEATURES
    else "-".join(BEST_FEATURES)
)

saved_paths = []

for rank, (name, config) in enumerate(
    FINAL_CONFIGS,
    start=1,
):
    tree_pred = np.quantile(
        tree_matrix_test,
        config["tree_q"],
        axis=1,
    )

    pair_pred = np.quantile(
        pair_matrix_test,
        config["pair_q"],
        axis=1,
    )

    blended = (
        (1.0 - config["pair_weight"])
        * tree_pred
        + config["pair_weight"]
        * pair_pred
    )

    final_pred = apply_postprocess(
        blended,
        config["round_decimals"],
        config["override_threshold"],
        nearest_dist_test,
        nearest_target_test,
    )

    submission = submission_template.copy()
    submission[TARGET] = final_pred

    filename = (
        f"submit_abtest_v8_rank{rank}_{name}"
        f"_features-{feature_label}.csv"
    )

    output_file = OUTPUT_DIR / filename

    submission.to_csv(
        output_file,
        index=False,
    )

    saved_paths.append(output_file)

    print("\nSaved:", output_file)
    print("Config:", config)
    display(submission.head())

print("\n생성 완료")
for path in saved_paths:
    print(path)



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_abtest_v8_rank1_cv_best_features-blood_pressure_ratio.csv
Config: {'tree_q': 0.54, 'pair_q': 0.48, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14748333333333333}


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.52
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_abtest_v8_rank2_safe_features-blood_pressure_ratio.csv
Config: {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.2, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14752}


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.49
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_abtest_v8_rank3_no_override_features-blood_pressure_ratio.csv
Config: {'tree_q': 0.5, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14756000000000002}


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_abtest_v8_rank4_tree_heavy_features-blood_pressure_ratio.csv
Config: {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14751666666666666}


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_abtest_v8_rank1_cv_best_features-blood_pressure_ratio.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_abtest_v8_rank2_safe_features-blood_pressure_ratio.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_abtest_v8_rank3_no_override_features-blood_pressure_ratio.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_abtest_v8_rank4_tree_heavy_features-blood_pressure_ratio.csv


## 확인해야 할 결과

1. `single_df`
   - 단일 피처 중 무엇이 Baseline보다 좋아졌는지 확인
2. `combination_df`
   - 단독 개선 피처끼리 조합했을 때 추가 개선이 있는지 확인
3. `search_df.head(30)`
   - 최종 Pair/quantile/override 포함 OOF 최적값 확인

단일 피처와 조합 피처 모두 개선하지 못하면 자동으로 Baseline을 유지합니다.
